# <font color="steelblue">Proyecto de clasificación — Variedades de dátil por visión artificial</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

## <font color="steelblue">Objetivos del proyecto</font>

A partir de **características extraídas por un sistema de visión artificial** (no de los píxeles), construir, comparar y **desplegar** un clasificador que distinga **7 variedades de dátil**. Este proyecto entrena competencias propias del **flujo de visión artificial basado en características**:

* **Importancia del escalado:** las variables están en **escalas muy dispares**; los modelos de distancia/lineales lo necesitan (demostradlo).
* **Redundancia y colinealidad de variables de ingeniería:** muchas características son **funciones deterministas** de otras (p. ej. `EQUIVALENT_DIAMETER` depende de `AREA`); conviene **reducir/seleccionar**.
* **Ablación por grupos de características:** las 34 variables se agrupan en **morfología, forma, color y textura-wavelet**. ¿Qué familia discrimina mejor la variedad?
* **Reflexión conceptual:** ventajas e inconvenientes del enfoque **basado en características** frente al **extremo-a-extremo (CNN)**.

Aplicaréis además el flujo completo: comparación de modelos, equilibrado, optimización, combinación, interpretación y despliegue.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto recoge **898 dátiles** fotografiados individualmente mediante un **sistema de visión
por computador** y clasificados en **7 variedades genéticas** (Koklu, Kursun, Taspinar & Cinar,
2021, *Mathematical Problems in Engineering*; distribuido en Kaggle como *Date Fruit Datasets*).
Las imágenes se capturaron en una **caja cerrada sin luz externa**, sobre **fondo verde**
(R=106, G=210, B=175) y se segmentaron por **umbralización de Otsu**.

Contiene **34 características numéricas** y una variable objetivo. **No hay valores faltantes**
y **no viene pre-dividido** en `train`/`test`. Se distribuye como `.xlsx` y `.arff`, **no como CSV**.

Cada fila corresponde a **un dátil**, y —esto es lo esencial— **no contiene píxeles**: contiene
**descriptores calculados a partir de la imagen ya segmentada**. Es decir, entre la fruta y el
modelo hay una capa de decisiones (umbralización, filtrado de fondo, definición de cada
descriptor) que el conjunto **no documenta fila a fila** y que no podéis auditar.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Morfología** *(miden tamaño; unidades en píxeles y píxeles²)*

| Variable | Tipo | Descripción |
|---|---|---|
| `AREA` | Entera | **Área** de la región segmentada. Órdenes de 10⁵. |
| `PERIMETER` | Numérica | **Perímetro** del contorno. |
| `MAJOR_AXIS` | Numérica | **Eje mayor** de la elipse ajustada. |
| `MINOR_AXIS` | Numérica | **Eje menor** de la elipse ajustada. |
| `EQDIASQ` | Numérica | **Diámetro equivalente**: diámetro del círculo de igual área. |
| `CONVEX_AREA` | Entera | Área de la **envolvente convexa**. |

**Bloque 2 — Forma** *(adimensionales; invariantes a escala)*

| Variable | Tipo | Descripción |
|---|---|---|
| `ECCENTRICITY` | Numérica | **Excentricidad** de la elipse ajustada (0 = círculo). |
| `SOLIDITY` | Numérica | `AREA / CONVEX_AREA`. **Concavidad** del contorno. Cerca de 1. |
| `EXTENT` | Numérica | `AREA` / área del rectángulo delimitador. |
| `ROUNDNESS` | Numérica | `4π·AREA / PERIMETER²`. **Circularidad**. |
| `ASPECT_RATIO` | Numérica | `MAJOR_AXIS / MINOR_AXIS`. **Elongación**. |
| `COMPACTNESS` | Numérica | `EQDIASQ / MAJOR_AXIS`. |
| `SHAPEFACTOR_1..4` | Numérica | Cuatro **factores de forma** derivados de ejes, área y perímetro. |

**Bloque 3 — Color** *(estadísticos por canal RGB; 5 momentos × 3 canales)*

| Familia | Variables | Descripción |
|---|---|---|
| Tendencia central | `MeanRR`, `MeanRG`, `MeanRB` | **Media** de intensidad por canal. |
| Dispersión | `StdDevRR`, `StdDevRG`, `StdDevRB` | **Desviación típica**. |
| Momentos altos | `SkewRR/RG/RB`, `KurtosisRR/RG/RB` | **Asimetría** y **curtosis** del histograma. |
| Información | `EntropyRR`, `EntropyRG`, `EntropyRB` | **Entropía** del canal. Valores **negativos y de magnitud enorme** (~10⁷): no están normalizados. |

**Bloque 4 — Textura**

| Variable | Tipo | Descripción |
|---|---|---|
| `ALLdaub4RR/RG/RB` | Numérica | Energía de la transformada **wavelet Daubechies-4** por canal. |

**Variable objetivo**

| Variable | Valores | Descripción |
|---|---|---|
| `Class` | `BERHI`, `DEGLET`, `DOKOL`, `IRAQI`, `ROTANA`, `SAFAVI`, `SOGAY` | **Variedad genética**. Nominal, **sin orden**. Reparto **desigual moderado**: de ~204 (`DOKOL`) a ~65 (`BERHI`), razón ≈ **3:1**. |

> ⚠️ **Los nombres del artículo no son los del fichero.** El paper habla de Barhee, Deglet Nour,
> Sukkary, Rotab Mozafati, Ruthana, Safawi y Sagai; la columna `Class` contiene los códigos en
> mayúsculas de arriba. **Comprobad la correspondencia con `value_counts()`** antes de dar por
> buena cualquier tabla de resultados publicada.

> **Limitación del dataset:** no hay identificador de **lote, sesión de captura, cosecha ni
> madurez**; tampoco peso, calibre real ni escala física (todo está en píxeles). No se puede
> saber si dos dátiles de la misma variedad se fotografiaron el mismo día. Tenlo en cuenta al
> interpretar.
> **Referencia de rendimiento:** los autores obtuvieron ~91 % (regresión logística), ~92,2 %
> (red neuronal) y ~92,8 % (*stacking*). Si superáis mucho esa cifra, sospechad de fuga.

### <font color="steelblue">Advertencias metodológicas</font>

1. **Las escalas difieren en siete órdenes de magnitud: escalar no es opcional.** `SOLIDITY`
   vive en ≈[0,98, 1], `AREA` en ≈10⁵ y las entropías en ≈−10⁷. Sin estandarizar, **kNN, SVM,
   PCA, k-medias y regresión regularizada quedan gobernados por `Entropy*` y `AREA`**, no porque
   sean informativas sino porque son grandes. Los árboles y sus ensambles son invariantes a
   transformaciones monótonas y no lo necesitan: esa asimetría es en sí misma un resultado que
   merece la pena mostrar en la comparativa.

2. **Escalad DENTRO del pipeline, nunca antes de partir.** Es el error más frecuente aquí. Si
   ajustáis el `StandardScaler` sobre el conjunto completo, la media y la desviación del *test*
   se filtran al entrenamiento: **fuga de datos**. Usad `Pipeline(StandardScaler(), modelo)` y
   dejad que la validación cruzada reajuste el escalador en **cada fold**. La diferencia de
   métricas suele ser pequeña, pero el hábito es lo que se evalúa.

3. **Colinealidad casi determinista: varias variables son funciones exactas de otras.**
   `EQDIASQ` es función de `AREA`; `SOLIDITY = AREA/CONVEX_AREA`; `ROUNDNESS = 4π·AREA/PERIMETER²`;
   `ASPECT_RATIO = MAJOR_AXIS/MINOR_AXIS`; `COMPACTNESS = EQDIASQ/MAJOR_AXIS`. No es correlación
   alta: es **redundancia algebraica**. Consecuencias: **VIF infinito**, coeficientes de regresión
   logística inestables y sin interpretación individual, e **importancias repartidas
   arbitrariamente** entre variables intercambiables (ninguna parecerá relevante aunque el tamaño
   sí lo sea). PCA sobre el bloque morfológico, o selección explícita, son respuestas razonables.

4. **El color mide la caja, no solo el dátil.** Iluminación fija, sin luz externa, fondo verde
   uniforme, misma cámara. Los estadísticos RGB codifican **la variedad y el sistema de captura
   a la vez**, y no hay forma de separarlos. Un modelo entrenado aquí **no funcionará con fotos
   de móvil**: es un caso de libro de **desplazamiento de dominio**. Además, si los dátiles de
   una misma variedad se fotografiaron en la misma sesión, el color puede estar capturando
   **la sesión** y no la genética — y sin identificador de lote **no podéis descartarlo**.

5. **La segmentación es una variable oculta.** `AREA`, `PERIMETER` y `CONVEX_AREA` dependen de
   dónde puso Otsu el umbral. Un halo de sombra mal filtrado infla el área; un brillo especular
   la reduce. Ese ruido **no es aleatorio**: correlaciona con el color del dátil (los `SAFAVI`,
   oscuros, se segmentan distinto que los `BERHI`, claros). O sea: **el error de medida de las
   variables morfológicas está correlacionado con la clase**. Es lo más sutil de este conjunto.

6. **`Skew*` y `Kurtosis*` son momentos de orden alto: extremadamente sensibles a atípicos.**
   Un solo píxel mal segmentado en la cola del histograma mueve la curtosis de forma visible.
   Inspeccionad su distribución antes de escalar y considerad `RobustScaler` o
   transformaciones de rango (`QuantileTransformer`) para ese bloque.

7. **Sin partición predefinida y con clases pequeñas.** 898 muestras, 7 clases, la minoritaria
   con ~65 ejemplos. Un 80/20 deja ~13 `BERHI` en test: cualquier métrica sobre esa clase tendrá
   un intervalo de confianza inservible. Usad **validación cruzada estratificada repetida**
   (p. ej. 5×10) y reportad la **desviación entre folds**, no un número aislado.

8. **34 dimensiones frente a 65 ejemplos en la clase minoritaria.** En la clase pequeña hay
   **el doble de características que de muestras**. Las distancias euclídeas se concentran
   (*maldición de la dimensionalidad*), lo que degrada kNN, y cualquier modelo flexible puede
   memorizar `BERHI` sin generalizar. Reducir dimensión **no es cosmética** aquí.

9. **Desequilibrio moderado: la exactitud global oculta el fallo donde importa.** `DOKOL` y
   `ROTANA` suman casi el 40 % de la muestra. Reportad **macro-F1** y la **matriz de confusión
   completa**: en la literatura, la confusión sistemática se da entre `DEGLET` y `SOGAY`, que
   son morfológicamente parecidos. Ese es el resultado interesante, no el *accuracy*.

10. **Ni causalidad ni consecuencias inocuas.** Que `AREA` tenga un SHAP alto **no significa**
    que el tamaño *determine* la variedad: significa que en **estas 898 fotos** el tamaño la
    separa. La variedad es genética; el tamaño es fenotipo, y depende de riego, cosecha y
    madurez, ninguna de las cuales está en el conjunto (**sesgo de variable omitida**). Y si
    el sistema se desplegara para **etiquetado comercial**, un falso positivo tiene coste real:
    vender `SOGAY` como `SAFAVI` es fraude alimentario. La métrica relevante no es la del
    cuaderno, sino la de la variedad que más se falsifica.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Escala dentro del `Pipeline`** (ajustado solo con el *train*): demuestra que importa para kNN/SVM/logística y que los árboles no lo necesitan.
2. **Trata la redundancia:** identifica la colinealidad y valora **selección de variables**/**PCA**; comenta su efecto.
3. **Ablación por grupos:** compara el poder predictivo de morfología, forma, color y textura por separado y en conjunto.
4. **Partición estratificada** (7 clases desiguales); el *test* solo se toca al final.
5. **Equilibrado solo en *train***; desequilibrio **moderado**.
6. **Métricas multiclase:** **macro-F1**/exactitud balanceada y **recall por clase** (no solo *accuracy*).
7. **Reproducibilidad y honestidad:** `random_state` fijado; reporta lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

> El dataset puede venir en **CSV o Excel**; el código contempla ambos (Excel requiere `openpyxl`).

In [ ]:
# !pip -q install kagglehub imbalanced-learn scikit-learn shap gradio openpyxl
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Paso 1: descargar el dataset
path = kagglehub.dataset_download("muratkokludataset/date-fruit-datasets")
print("Ruta:", path, "| Archivos:", os.listdir(path))
archivos = os.listdir(path)

# Paso 2: cargar CSV o XLSX según la versión descargada
archivo_csv  = [f for f in archivos if f.endswith('.csv')]
archivo_xlsx = [f for f in archivos if f.endswith('.xlsx')]
if archivo_csv:
    df = pd.read_csv(os.path.join(path, archivo_csv[0]))
elif archivo_xlsx:
    df = pd.read_excel(os.path.join(path, archivo_xlsx[0]))
else:
    raise FileNotFoundError("No se encontró CSV ni XLSX.")
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

**Tareas obligatorias**
1. **Objetivo.** Distribución de `Class` (7 variedades): constatad el **desequilibrio moderado**.
2. **Grupos de características (clave).** Construid las listas de las cuatro familias (**morfología, forma, color, textura**) a partir de la descripción.
3. **Escalas.** Comparad rangos (`AREA` vs `ECCENTRICITY`…): justificad por qué hará falta **escalar**.
4. **Colinealidad.** Matriz de correlación: localizad los bloques muy correlacionados (morfología; algunas de forma). ¿Qué variables son casi deterministas de otras?
5. **Separabilidad.** Proyectad con **PCA** y **t-SNE** coloreando por variedad: ¿se separan? ¿qué variedades se confunden?
6. **Conclusión:** 3–4 hallazgos.

# <font color="steelblue">Fase 2 — Preprocesado y partición</font>

**Tareas obligatorias**
1. **Objetivo:** codificad `Class` (7 clases). `X` = las 34 características.
2. **Partición estratificada** (clases desiguales).
3. **Escalado** dentro del `Pipeline` (imprescindible para logística/SVM/kNN; los árboles no lo necesitan).
4. **(Opcional) Redundancia:** valorad **PCA** o **selección de variables** para los modelos lineales/de distancia, y comparad con usar las 34.
5. Guardad las **listas de grupos** (Fase 1) para la **ablación** de la Fase 7.

> **A responder:** ¿por qué `AREA`, `CONVEX_AREA` y `EQUIVALENT_DIAMETER` aportan información muy solapada?

# <font color="steelblue">Fase 3 — Modelos base, comparación y efecto del escalado</font>

**Tareas obligatorias**
1. Comparad **≥5 familias** del curso: **Regresión logística (multinomial)**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. **Demostración del escalado (clave).** Entrenad **kNN** y/o **SVM** **con** y **sin** escalado y mostrad la diferencia; comprobad que el **Random Forest** apenas cambia. Es la lección de estos datos.
3. **CV repetida** estratificada (dataset pequeño) con **`f1_macro`**/exactitud balanceada.
4. **Tabla** comparativa + comentario.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

El desequilibrio es **moderado** (de 65 a 204 por clase): medid el efecto de tratarlo (material **11**) sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'`.
3. **Sobremuestreo:** **SMOTE** (multiclase; dentro del `ImbPipeline`).
4. (Opcional) submuestreo/híbrido.

Reportad **macro-F1** y **recall por clase** (en especial las minoritarias, p. ej. `Barhee`) y razonad la mejor opción.

> **Sin fugas:** remuestreo dentro de `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y `f1_macro`; búsqueda **sobre el `Pipeline`** (prefijo `clf__`; si usas PCA, también `pca__n_components`).
3. (Recomendado por el n moderado) **CV anidada**.
4. Reportad mejores hiperparámetros y la mejora.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual** (macro-F1): ¿mejora?
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, ablación por grupos e interpretación</font>

Esta es la fase **distintiva**. El *test* se usa una sola vez.

**Tareas obligatorias**
1. **Métricas finales:** **matriz de confusión** 7×7, `classification_report`, **macro-F1**, **recall por clase**. ¿Qué variedades se confunden entre sí?
2. **Ablación por grupos de características (clave).** Entrenad y validad el mejor modelo usando **solo morfología**, **solo forma**, **solo color**, **solo textura** y **todas**. Comparad el macro-F1: **¿qué familia de descriptores discrimina mejor** la variedad? ¿El color/tamaño basta o hacen falta forma/textura?
3. **Interpretabilidad (SHAP / importancia).** ¿Qué descriptores concretos pesan más? ¿Es coherente con lo visto en la ablación?
4. **Reflexión características vs CNN.** Discutid el compromiso: el enfoque por características es **interpretable y eficiente**; una CNN sobre imágenes podría captar texturas más finas a costa de interpretabilidad y datos.
5. **Discusión crítica:** un solo sistema de captura/iluminación, generalización a otras condiciones, variedades no representadas.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** (escalado + modelo) con `joblib`.
2. **Función de predicción:** `predecir_variedad(...)` que tome las **34 características** de un dátil y devuelva la variedad y las **probabilidades**.
3. **Interfaz:** app **Gradio** (o `ipywidgets`) que reciba las características (o suba un CSV con una muestra) y muestre la variedad predicha. En Colab da un **enlace público** (incluidlo).
4. (Opcional, nota extra) Esbozad cómo sería el **sistema completo de visión**: imagen → segmentación → extracción de características → este modelo. (Una **CNN** sería el enfoque alternativo extremo-a-extremo.)

> **Aviso:** herramienta **educativa**; el modelo depende del **sistema de captura** (cámara, iluminación, fondo) con el que se extrajeron las características y puede no generalizar a otras condiciones.

# <font color="steelblue">Pistas y errores típicos</font>

* **Escala (dentro del `Pipeline`).** Con escalas que van de [0,1] a miles de píxeles, kNN/SVM/logística **fracasan sin escalar**; los árboles no lo necesitan. Demuéstralo.
* **Redundancia.** Muchas características son funciones de otras (área↔diámetro equivalente↔convex area): considera **PCA**/selección para modelos lineales.
* **Ablación reveladora.** Comparar morfología vs forma vs color vs textura te dice **qué información** distingue de verdad las variedades (a menudo el color y el tamaño mandan, pero compruébalo).
* **Multiclase desigual:** usa **macro-F1** y recall por clase, no solo *accuracy*.
* **Características vs CNN:** este flujo es interpretable y barato; una CNN podría ganar en textura fina a costa de datos e interpretabilidad.
* **Despliegue:** guarda el **Pipeline entero**; el modelo depende del sistema de captura.

# <font color="steelblue">Referencias</font>

* Koklu, M., Kursun, R., Taspinar, Y. S. & Cinar, I. (2021). *Classification of Date Fruits into Genetic Varieties Using Image Analysis*. Mathematical Problems in Engineering (Hindawi).
* Cuadernos del curso: *kNN*, *SVM*, *Random Forest*, *Boosting*, *Componentes principales y variantes*, *Métodos de manifold*, *Equilibrando las muestras*.